

This script does not replace finalization_and_mapping.py as that serves a different purpose. However, they occur after the same step.

The main functions of this script include:


1.   Loading HawkEars Scores & Confirmed Positions from detections.csv file and from shelve file
2.   Extract which ARUs heard each detection, counts how many, and stores the ARU IDs.
3. Creates weml_confirmed_locations_tessa.csv with additional columns which will be helpful for the following scripts.
4. Creates a time-animated map (period='PT1M' — time steps of 1 minute
duration='PT5M' — each point visible for 5 minutes).

In [ ]:
import pandas as pd
import numpy as np
import shelve
import json
import folium
from folium.plugins import TimestampedGeoJson
from pyproj import Transformer
from pathlib import Path

# CLASS DEFINITION
class CsvPositionEstimate:
    def __init__(self, row):
        self.class_name = row["class_name"]
        self.start_timestamp = row["start_timestamp"]
        self.duration = float(row["duration"])
        self.mean_residual = float(row["mean_residual"])
        self.residual_rms = float(row["residual_rms"])
        self.mean_cc_max = float(row["mean_cc_max"])
        self.location_estimate = np.array([row["pred_x"], row["pred_y"], row["pred_z"]])
        self.receiver_files = json.loads(row["receiver_files"])
        self.receiver_locations = np.array(json.loads(row["receiver_locations"]))
        self.receiver_start_time_offsets = np.array(json.loads(row["receiver_start_time_offsets"]))
        self.tdoas = np.array(json.loads(row["tdoas"]))
        self.cc_maxs = np.array(json.loads(row["cc_maxs"]))
        self.distance_residuals = np.array(json.loads(row["distance_residuals"]))

# --- CONFIG ---
scores_path = '/media/UofA/BU_Work/BayneLabWorkSpace/Katrine_workspace/Localization_per_day/OKLG-8-20250615/minspec_output/detections.csv'
shelf_path = '/media/UofA/BU_Work/BayneLabWorkSpace/Katrine_workspace/Localization_per_day/OKLG-8-20250615/weml_confirmed.out'
output_csv = '/media/UofA/BU_Work/BayneLabWorkSpace/Katrine_workspace/Localization_per_day/OKLG-8-20250615/weml_confirmed_locations_tessa.csv'
output_html = '/media/UofA/BU_Work/BayneLabWorkSpace/Katrine_workspace/Localization_per_day/OKLG-8-20250615/weml_time_slider_tessa.html'
# FIX: Consistent variable name for ARU coords
aru_coords_path = '/media/UofA/BU_Work/BayneLabWorkSpace/Katrine_workspace/AudioMothSync/RTK_Coordinates - BC Alberts (EPSG-2955).csv'

# 1. SCORE DISTRIBUTION
print("=" * 50); print("HawkEars Score Distribution"); print("=" * 50)
scores = pd.read_csv(scores_path)
print(scores['score'].describe())

# 2. LOAD POSITIONS
with shelve.open(shelf_path, 'r') as db:
    positions = db['position_estimates']
print(f"\nConfirmed Positions: {len(positions)}")

# 3. COORDINATE STATS
coords = [(p.location_estimate[0], p.location_estimate[1], p.location_estimate[2]) for p in positions]
df_utm = pd.DataFrame(coords, columns=['x', 'y', 'z'])
print("\nUTM Coordinates summary calculated.")

# Parse receiver file paths to extract ARU device IDs
def extract_aru_ids(receiver_files_json):
    """Pull ARU device ID from each receiver filepath."""
    files = json.loads(receiver_files_json)
    ids = []
    for f in files:
        # Filename pattern: .../OKLG-8-XXXX/... or device id in folder name
        # Adjust this split to match your actual folder naming convention
        parts = Path(f).parts
        for part in parts:
            if 'OKLG' in part or part.isdigit():
                ids.append(part)
                break
    return ids

# 4. EXPORT TO CSV
export_data = []
for p in positions:
    # Extract which ARUs heard this detection
    aru_ids = [Path(f).parent.name.split('-')[-1] for f in p.receiver_files]
    n_arus    = len(aru_ids)

    export_data.append({
        'timestamp':        p.start_timestamp,
        'x':                p.location_estimate[0],
        'y':                p.location_estimate[1],
        'z':                p.location_estimate[2],
        'residual_rms':     p.residual_rms,
        'mean_cc_max':      p.mean_cc_max,
        'n_receivers':      n_arus,                        # ← how many ARUs heard it
        'receiver_ids':     json.dumps(aru_ids),           # ← which ARU IDs
        'receiver_files':   json.dumps(p.receiver_files),  # ← full paths
        'receiver_locations': json.dumps(p.receiver_locations.tolist()),
    })

# Sanity check — print first 3 detections' receiver IDs
print("\nSample receiver_ids (should match ARU device IDs in coords CSV):")
for row in export_data[:3]:
    print(f"  {row['receiver_ids']}")

# Also print what IDs exist in your ARU coords CSV for comparison
df_arus = pd.read_csv(aru_coords_path)
df_grid = df_arus[df_arus['localization_grid'] == 'OKLG-8'].copy()
df_grid['device_id_short'] = df_grid['device_id'].str.split('-').str[-1]
print(f"\nARU IDs in coords CSV: {sorted(df_grid['device_id_short'].tolist())}")


df_export = pd.DataFrame(export_data)
df_export.to_csv(output_csv, index=False)
print(f"✓ CSV exported to: {output_csv}")

# 5. MAPPING
print("\nCreating interactive timeline map...")
transformer = Transformer.from_crs("EPSG:2955", "EPSG:4326", always_xy=True)
features = []

for p in positions:
    # OPTIONAL: Filter out high-error points for a cleaner map
    if p.residual_rms > 5.0: continue

    lon, lat = transformer.transform(p.location_estimate[0], p.location_estimate[1])
    local_dt = pd.to_datetime(p.start_timestamp) - pd.Timedelta(hours=7)
    orig_file = Path(p.receiver_files[0]).name if p.receiver_files else "Unknown"

    features.append({
        'type': 'Feature',
        'geometry': {'type': 'Point', 'coordinates': [lon, lat]},
        'properties': {
            'time': local_dt.isoformat(),
            'style': {'color': 'red', 'fillColor': 'orange', 'radius': 8, 'fillOpacity': 0.7},
            'popup': f"<b>Time:</b> {local_dt.strftime('%H:%M:%S')}<br><b>RMS:</b> {p.residual_rms:.2f}m<br><b>File:</b> {orig_file}"
        }
    })

# Initialize Map centered on median coordinates
m = folium.Map(location=[df_utm['y'].median(), df_utm['x'].median()], zoom_start=18) # Note: median lats/lons usually better but medians of x/y work here if center is similar
# Better: center using transformed coords
center_lon, center_lat = transformer.transform(df_utm['x'].median(), df_utm['y'].median())
m = folium.Map(location=[center_lat, center_lon], zoom_start=18)

folium.TileLayer(tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
                 attr='Esri', name='Satellite').add_to(m)

# Add ARU locations
df_arus = pd.read_csv(aru_coords_path)
df_grid = df_arus[df_arus['localization_grid'] == 'OKLG-8'].copy()
for _, row in df_grid.iterrows():
    a_lon, a_lat = transformer.transform(row['ground_truth_easting'], row['ground_truth_northing'])
    folium.Marker([a_lat, a_lon],
                  popup=f"ARU: {row['device_id']}",
                  icon=folium.Icon(color='black', icon='microphone', prefix='fa')).add_to(m)

# Add Timeline
TimestampedGeoJson(
    {'type': 'FeatureCollection', 'features': features},
    period='PT1M', duration='PT5M',
    add_last_point=True, auto_play=False, date_options='HH:mm:ss'
).add_to(m)

m.save(output_html)
print(f"Success! Map saved to: {output_html}")
